# Глава 18. LLM на практике: промптинг и классификация текста

_Ноутбук сгенерирован из учебника «Машинное обучение для начинающих». Запускайте ячейки по порядку сверху вниз в Google Colab._

**Цель главы  🎯**

После этой главы ты сможешь сформулировать эффективный промпт с ролью и примерами, решить реальную задачу через API (классификация отзывов по тональности) и критически оценить ответ модели.

Теперь применим LLM к настоящей задаче. И это будет не «ещё один чат-бот»: мы научимся управлять моделью через грамотные промпты и решим реальную бизнес-задачу — автоматический анализ тональности отзывов. Такие системы реально работают в магазинах, банках и сервисах, разбирая тысячи сообщений клиентов.

## Промпт — это инструкция для модели

**Аналогия: задание для стажёра  💡**

Промпт (prompt) — это то, что ты пишешь модели. Представь, что даёшь задание толковому, но незнакомому стажёру: он способный, но не читает мысли. Чем яснее и подробнее объяснишь, тем лучше результат. «Сделай нормально» — плохое задание, стажёр не поймёт. «Определи тональность отзыва: ответь одним словом — позитив или негатив» — хорошее, понятное задание. Качество ответа модели напрямую зависит от качества твоего промпта. Промпт-инжиниринг — это искусство ясно ставить задачу.

## Три приёма хорошего промпта

### 1. Дай роль

Начни промпт с назначения роли: «Ты — опытный редактор», «Ты — строгий учитель математики». Роль настраивает модель на нужный стиль, тон и уровень ответа. Это как сказать стажёру, в качестве кого он сейчас работает: от этого зависит, как он подойдёт к задаче.

### 2. Приведи примеры

Покажи модели 2–3 примера «вход → правильный ответ». Модель мгновенно схватывает образец и повторяет его на новых данных. Этот приём называют few-shot («несколько примеров») — один из самых мощных и простых способов резко улучшить качество ответов, особенно в задачах классификации.

### 3. Попроси рассуждать по шагам

Для сложных задач добавь «Объясни ход мысли шаг за шагом». Модель, «проговаривая» рассуждение вслух, реже ошибается — как ученик, который решает задачу на бумаге, а не в уме. Приём называют цепочкой мыслей (chain of thought) и применяют в задачах, где нужен многошаговый вывод (математика, логика).

Приём

Плохой промпт

Хороший промпт

Роль

Проверь текст

Ты редактор. Найди ошибки в тексте

Примеры

Определи тон

«Супер!» → позитив. «Ужас» → негатив. «Нормально» → ?

Формат

Расскажи про отзыв

Ответь ОДНИМ словом: позитив или негатив

## Подключаемся к API

Чтобы обращаться к модели из кода (а не через сайт), используют API — программный доступ к модели по секретному ключу. Ключ выдаётся в личном кабинете провайдера (OpenAI, Anthropic и другие). Ниже — пример для Anthropic API; для OpenAI код очень похож по смыслу, меняются лишь названия.

**Внимание: береги ключ как пароль  ⚠️**

API-ключ — это как пароль от банковской карты, привязанной к твоему счёту. Никогда не публикуй его в открытом коде, не выкладывай на GitHub и не показывай в презентации на экране. В Colab храни ключ в разделе «Секреты» (значок ключа на левой панели), а не прямо в ячейке кода. За утёкший ключ, которым воспользуются чужие люди, платить будешь ты. Это правило номер один при работе с любым API.

In [ ]:
# Запускать в Google Colab
!pip install anthropic -q

import anthropic
from google.colab import userdata   # безопасное хранилище ключей

# Ключ берётся из "Секретов" Colab, а не пишется в коде
client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

def ask(prompt):
    msg = client.messages.create(
        model="claude-3-5-haiku-latest",
        max_tokens=50,
        messages=[{"role": "user", "content": prompt}],
    )
    return msg.content[0].text

print(ask("Ответь одним словом: столица Франции?"))

_Ожидаемый результат: Париж._

**Полезно знать  ℹ️**

Имя модели (здесь claude-3-5-haiku-latest) со временем меняется — провайдеры выпускают новые версии. Актуальное имя всегда смотри в документации провайдера (Anthropic, OpenAI и др.); если код не работает, первым делом проверь, не устарело ли название модели.

## Главный кейс: анализ тональности отзывов

Это настоящая бизнес-задача. Представь: у интернет-магазина тысячи отзывов, и менеджеру надо быстро понять, каких больше — довольных или недовольных, чтобы вовремя среагировать. Вручную читать тысячи отзывов невозможно. Сделаем классификатор тональности на промптах — и он разметит их за секунды.

In [ ]:
reviews = [
    "Отличный товар, всем рекомендую!",
    "Пришло сломанное, ужасный сервис.",
    "Ну, в целом нормально, ничего особенного.",
    "Быстрая доставка, качество супер!",
]

def sentiment(review):
    prompt = (
        "Ты анализируешь отзывы. Определи тональность.\n"
        "Ответь ОДНИМ словом: позитив, негатив или нейтрально.\n\n"
        "Примеры:\n"
        "«Всё супер» -> позитив\n"
        "«Кошмар, деньги на ветер» -> негатив\n\n"
        f"Отзыв: «{review}»"
    )
    return ask(prompt).strip()

for r in reviews:
    print(f"{sentiment(r):12} | {r}")

_Ожидаемый результат: у каждого отзыва проставлена тональность._

Обрати внимание на важнейший момент: мы НЕ обучали модель. Никакого train/test, никакого .fit(). Модель уже умеет понимать язык — мы лишь грамотно объяснили задачу промптом с ролью и примерами. В этом сила LLM: одна и та же модель решает тысячи разных задач без переобучения, если правильно поставить вопрос. Сравни с фазой 2, где под каждую задачу мы обучали отдельную модель.

## Бот с характером

Роль в промпте задаёт не только точность, но и характер. Меняя системную инструкцию (system), ты получаешь бота с любой личностью — от строгого учителя до весёлого пирата. Системная инструкция — это «настройка характера», которая действует на весь разговор. Попробуй и убедись, как сильно роль меняет тон одних и тех же ответов.

In [ ]:
def bot(user_message, character):
    msg = client.messages.create(
        model="claude-3-5-haiku-latest",
        max_tokens=200,
        system=character,   # характер бота задаётся здесь
        messages=[{"role": "user", "content": user_message}],
    )
    return msg.content[0].text

pirate = "Ты пират. Отвечай как морской волк, с 'йо-хо-хо'."
print(bot("Расскажи, что такое машинное обучение", pirate))

## Для сильных учеников: мини-RAG по тексту

**Аналогия: экзамен с открытой книгой  💡**

Обычная LLM отвечает по памяти и может галлюцинировать (глава 17). RAG (Retrieval-Augmented Generation) даёт модели «шпаргалку»: сначала мы находим нужный кусок из своего документа, затем передаём его модели вместе с вопросом. Это как экзамен с открытой книгой — модель отвечает не по памяти, а по конкретному тексту перед ней, и врёт гораздо меньше. Так работают ИИ-помощники по документам компаний.

In [ ]:
# Мини-RAG "на пальцах": вставляем нужный текст прямо в промпт
document = """
Курс длится 3 месяца и состоит из 24 занятий.
Финальный проект защищается на Demo Day перед родителями.
"""

question = "Сколько занятий в курсе?"

prompt = (
    "Ответь на вопрос ТОЛЬКО по тексту ниже. "
    "Если ответа в тексте нет — так и скажи, не выдумывай.\n\n"
    f"Текст:\n{document}\n\n"
    f"Вопрос: {question}"
)
print(ask(prompt))

Это упрощённый RAG: настоящие системы автоматически ищут нужный фрагмент среди тысяч страниц по эмбеддингам (тем самым, из главы 17 — близкие по смыслу куски находятся как соседи на карте смыслов). Но идея та же и главная — заземлить ответ модели на реальный проверенный текст, а не на «память» модели. Инструкция «не выдумывай» — важная часть промпта против галлюцинаций.

## Конкурс: лучший классификатор

Устройте в классе соревнование: у кого промпт точнее размечает общий набор из 20 отзывов? Побеждает не самый длинный промпт, а самый продуманный — с чёткой ролью, хорошими примерами и ясным форматом ответа. Это отличный способ на практике почувствовать, что промпт-инжиниринг — реальный навык, а не магия.

**Главная мысль фазы 4  🎯**

Ты понял, как LLM превращает текст в токены и эмбеддинги, как работает внимание и почему модель галлюцинирует. И ты освоил главный прикладной навык эпохи ИИ — управлять моделью через промпты и решать реальные задачи через API, критически оценивая ответы. Ты научился заставлять готовую модель работать на тебя без всякого обучения. Дальше мы научимся ускорять создание моделей с AutoML и выкладывать приложения в интернет.

## Задания

Базовый уровень

Собери классификатор тональности на 10 своих отзывов (можно реальных — из магазина или отзывов на фильм). Улучши промпт, добавив роль и 3 примера. Стало ли меньше ошибок? Запиши до/после.

Со звёздочкой  ⭐

Сделай мини-RAG по тексту любимой статьи или главы учебника: вставь текст в промпт и задай 3 вопроса. Затем задай эти же вопросы БЕЗ текста и сравни — где модель галлюцинирует? Опиши, как RAG изменил ответы.

Вопросы для обсуждения  💬

Чем управление LLM через промпт принципиально отличается от обучения модели в фазе 2?

Почему API-ключ нельзя публиковать в коде — что может случиться?

Как RAG уменьшает галлюцинации? Почему он не убирает их полностью?

**Проверь себя  ℹ️**

1. Назови три приёма хорошего промпта.

2. Почему при работе с LLM не нужен train/test, как в фазе 2?

3. Как RAG помогает модели отвечать точнее?

ФАЗА 5

AutoML и AI-инструменты

_Ускоряем работу готовыми инструментами и выкладываем приложение в интернет._

В фазе 2 ты собирал модели руками — и это бесценный навык. Теперь узнаешь, как получить хороший результат быстро: инструменты AutoML сами перебирают модели и настройки, а Hugging Face даёт тысячи готовых моделей. И, главное, ты научишься выкладывать приложение в интернет по постоянному адресу — превращать код в продукт, которым может пользоваться кто угодно.